<!-- Notebook A banner -->
<div style="background:#E1F5EE;border-left:5px solid #1D9E75;border-radius:0 12px 12px 0;
            padding:22px 28px 18px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
            margin-bottom:4px">
  <p style="margin:0 0 4px;font-size:11px;font-weight:500;letter-spacing:.08em;
             color:#1D9E75;text-transform:uppercase">
    2nd International Summer School on AI for Diabetes Management &nbsp;·&nbsp; University of Girona &nbsp;·&nbsp; Day 3 &nbsp;·&nbsp; 13:00–15:00
  </p>
  <h1 style="margin:6px 0 4px;font-size:24px;font-weight:600;color:#04342C;line-height:1.25">
    From Population Models to Personalized Digital Twins
  </h1>
  <p style="margin:0;font-size:16px;font-weight:400;color:#1D9E75">
    Adapting AI Models to Individuals
  </p>
  <p style="margin:12px 0 0;font-size:13.5px;color:#0F6E56;line-height:1.5;font-style:italic">
    Notebook A &nbsp;·&nbsp; build the conditional VAE and then confirm it matches the population-pretrained model.
  </p>
  <p style="margin:14px 0 0;font-size:13px;color:#0F6E56">
    <b>Notebook A</b> &nbsp;·&nbsp; Dr. Omer Mujahid &nbsp;·&nbsp; UdG–Dexcom Chair for Advancing AI in Clinical Practice
  </p>
</div>

### Setup
Run this first. It calls the required libraries.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # route tf.keras -> Keras 2 (must precede TF import)

import os
REPO = "diabetes-digital-twins-workshop"
if not os.path.exists(f"/content/{REPO}"):
    !git clone -q https://github.com/omermujahid1/{REPO}.git /content/{REPO}
%cd /content/{REPO}

import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Model, Input
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.layers import (Dense, Dropout, LeakyReLU, Flatten, Lambda,
                                      LSTM, Reshape, Concatenate, TimeDistributed)
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model

from twin_workshop import (HORIZON, LATENT_DIM, HIDDEN, IN_SHAPE, OUT_SHAPE,
                           MIN_GAIN_MGDL, CONTROL_SCALE, PRETRAINED,
                           list_patients, load_patient, make_windows, holdout_split,
                           verify_setup)
verify_setup()

/content/diabetes-digital-twins-workshop
Workshop setup check
----------------------------------------
  TensorFlow 2.20.0  | legacy Keras: True
  data folder: 1 patients found (csv)


/usr/local/lib/python3.12/dist-packages/tf_keras/src/initializers/initializers.py:121: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


  pretrained decoder loads + forward pass OK
----------------------------------------
READY


True

## 1 · The encoder  q(z | bg, insulin, carbs)

Each input is projected to width 18 and stacked into a 3-step sequence `[insulin, carbs, bg]`, read by one recurrent layer into `z_mean` / `z_log_var`.

<!-- A1 — before cell 9 -->
<div style="border:1.5px dashed #0F6E56;border-radius:14px;background:#FDF0EA;padding:18px 22px;margin:16px 0">
<div style="display:flex;align-items:center;gap:9px;margin-bottom:10px">
<span style="width:22px;height:22px;border-radius:50%;background:#0F6E56;color:#E1F5EE;font-size:12px;font-weight:700;display:inline-flex;align-items:center;justify-content:center;flex-shrink:0">?</span>
<span style="color:#0F6E56;font-size:11px;font-weight:600;letter-spacing:.09em;text-transform:uppercase">Before you run this</span>
</div>
<p style="margin:0;color:#04342C;font-size:15.5px;font-weight:600;font-style:italic;line-height:1.45">Why one sequence, not three encoders?</p>
<p style="margin:10px 0 0;color:#063d31;font-size:13.5px;line-height:1.65">Insulin, carbs, and BG each get their own Dense projection — but then they're stacked into <i>one</i> (3, 18) sequence and read by a single LSTM, rather than encoding each signal separately and concatenating the results. What can a single recurrent layer reading all three stacked together learn that three independent encoders, combined only at the end, could not?</p>
<p style="margin:14px 0 0;padding-top:12px;border-top:1px dashed #0F6E56;color:#0F6E56;font-size:13.5px;line-height:1.5"><b>Think about →</b> the LSTM reads the sequence in a fixed order — insulin, then carbs, then BG. Does that order matter, or is it arbitrary?</p>
</div>

<div style="border-left:4px solid #1D9E75;background:#E1F5EE;padding:14px 18px;border-radius:0 8px 8px 0">
<div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
<span style="background:#0F6E56;color:#E1F5EE;font-size:11px;font-weight:500;letter-spacing:.04em;padding:3px 9px;border-radius:20px">BLANK 1</span>
<span style="color:#04342C;font-size:15px;font-weight:500">Recurrent layer</span>
<span style="margin-left:auto;color:#0F6E56;font-size:12px">Notebook A · encoder</span>
</div>
<p style="margin:6px 0 0;color:#063d31;font-size:13.5px;line-height:1.6">The stacked <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">(3, 18)</code> sequence must be read into a single vector. Use a recurrent layer of width 100 that returns only its final state (not the full sequence).</p>
<p style="margin:8px 0 0;color:#0F6E56;font-size:13.5px"><b style="font-weight:500">Your task:</b> call the recurrent layer on <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">merged</code>, width 100, final state only.</p>
</div>

<div style="border-left:4px solid #1D9E75;background:#E1F5EE;padding:14px 18px;border-radius:0 8px 8px 0">
<div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
<span style="background:#0F6E56;color:#E1F5EE;font-size:11px;font-weight:500;letter-spacing:.04em;padding:3px 9px;border-radius:20px">BLANK 2</span>
<span style="color:#04342C;font-size:15px;font-weight:500">Reparameterization trick</span>
<span style="margin-left:auto;color:#0F6E56;font-size:12px">Notebook A · encoder</span>
</div>
<p style="margin:6px 0 0;color:#063d31;font-size:13.5px;line-height:1.6">You can't backpropagate through a random draw. Rewrite a sample from <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">N(z_mean, z_var)</code> as fixed noise scaled by the standard deviation, so gradients still reach <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">z_mean</code> and <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">z_log_var</code>.</p>
<p style="margin:8px 0 0;color:#0F6E56;font-size:13.5px"><b style="font-weight:500">Your task:</b> combine the mean, log-variance, and <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">eps</code> into one differentiable expression.</p>
</div>

In [5]:
def define_encoder():
    init = RandomNormal(stddev=0.02)
    in_bg      = Input(shape=OUT_SHAPE, dtype=tf.float32, name="in_bg")
    in_insulin = Input(shape=IN_SHAPE,  dtype=tf.float32, name="in_insulin")
    in_carbs   = Input(shape=IN_SHAPE,  dtype=tf.float32, name="in_carbs")

    n = HORIZON
    bg_r  = Reshape((1, n))(Dense(n, kernel_initializer=init)(in_bg))
    ins_r = Reshape((1, n))(Dense(n, kernel_initializer=init)(in_insulin))
    car_r = Reshape((1, n))(Dense(n, kernel_initializer=init)(in_carbs))
    merged = Concatenate(axis=1)([ins_r, car_r, bg_r])              # (3, 18)

    # ---- BLANK 1 ----
    #e = LSTM(units=100, return_sequences=False)(merged)       # hint: LSTM, width 100, return_sequences=False
    e = LSTM(100, return_sequences=False, kernel_initializer=init)(merged)       # hint: LSTM, width 100, return_sequences=False

    z_mean    = Dense(LATENT_DIM, name="z_mean")(e)
    z_log_var = Dense(LATENT_DIM, name="z_log_var")(e)

    def sampling(args):
        zm, zv = args
        eps = tf.random.normal(shape=(tf.shape(zm)[0], tf.shape(zm)[1]))
        # ---- BLANK 2 ----
        return zm + tf.exp(0.5 * zv) * eps            # hint: zm + std * eps, std = exp(0.5 * zv)

    z = Lambda(sampling, output_shape=(LATENT_DIM,), name="z")([z_mean, z_log_var])
    return Model([in_bg, in_insulin, in_carbs], [z_mean, z_log_var, z], name="encoder")

encoder = define_encoder(); encoder.summary()

/usr/local/lib/python3.12/dist-packages/tf_keras/src/initializers/initializers.py:121: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


Model: "encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 in_insulin (InputLayer)     [(None, 1, 1)]               0         []                            
                                                                                                  
 in_carbs (InputLayer)       [(None, 1, 1)]               0         []                            
                                                                                                  
 in_bg (InputLayer)          [(None, 1, 18)]              0         []                            
                                                                                                  
 dense_10 (Dense)            (None, 1, 18)                36        ['in_insulin[0][0]']          
                                                                                            

## 2 · The decoder — base trajectory + sign-constrained control

The output is `base(z) + dose-weighted control`. The control weights pass through `softplus` (positive magnitude); the **sign** is then forced so insulin can only lower BG and carbs can only raise it.

<div style="border-left:4px solid #1D9E75;background:#E1F5EE;padding:14px 18px;border-radius:0 8px 8px 0">
<div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
<span style="background:#0F6E56;color:#E1F5EE;font-size:11px;font-weight:500;letter-spacing:.04em;padding:3px 9px;border-radius:20px">BLANK 3</span>
<span style="color:#04342C;font-size:15px;font-weight:500">Monotonicity constraint</span>
<span style="margin-left:auto;color:#0F6E56;font-size:12px">Notebook A · decoder</span>
</div>
<p style="margin:6px 0 0;color:#063d31;font-size:13.5px;line-height:1.6">Both weight streams come out of <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">softplus(...) + MIN_GAIN_MGDL</code> as positive magnitudes. Physiology requires the insulin weight to be strictly negative (more insulin lowers BG) while the carb weight stays positive. The carb line is given; write the insulin line.</p>
<p style="margin:8px 0 0;color:#0F6E56;font-size:13.5px"><b style="font-weight:500">Your task:</b> force <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">w_ins</code> negative by negating the positive magnitude.</p>
</div>

<!-- A2 — before cell 12 -->
<div style="border:1.5px dashed #0F6E56;border-radius:14px;background:#FDF0EA;padding:18px 22px;margin:16px 0">
<div style="display:flex;align-items:center;gap:9px;margin-bottom:10px">
<span style="width:22px;height:22px;border-radius:50%;background:#0F6E56;color:#E1F5EE;font-size:12px;font-weight:700;display:inline-flex;align-items:center;justify-content:center;flex-shrink:0">?</span>
<span style="color:#0F6E56;font-size:11px;font-weight:600;letter-spacing:.09em;text-transform:uppercase">Before you run this</span>
</div>
<p style="margin:0;color:#04342C;font-size:15.5px;font-weight:600;font-style:italic;line-height:1.45">What would an unconstrained model learn?</p>
<p style="margin:10px 0 0;color:#063d31;font-size:13.5px;line-height:1.65">You're about to force the insulin weight negative by construction, so the model cannot learn that more insulin raises BG, no matter what the training data looks like. Suppose you removed that constraint. In real CGM logs, insulin doses and meals often cluster in time. Could the model end up learning the <i>wrong</i> sign for insulin just by picking up that correlation? What would that mean for anyone later using this model to simulate "what if I gave more insulin"?</p>
<p style="margin:14px 0 0;padding-top:12px;border-top:1px dashed #0F6E56;color:#0F6E56;font-size:13.5px;line-height:1.5"><b>Keep this in mind →</b> you'll see exactly this scenario tested in Notebook B's counterfactual section.</p>
</div>

In [6]:
def define_decoder():
    init = RandomNormal(stddev=0.02)
    seq_len, hidden = HORIZON, HIDDEN
    in_lat  = Input(shape=(LATENT_DIM,), dtype=tf.float32, name="in_lat")
    in_ins  = Input(shape=IN_SHAPE,      dtype=tf.float32, name="in_ins")
    in_carb = Input(shape=IN_SHAPE,      dtype=tf.float32, name="in_carb")

    # base branch (trajectory shape from z)
    h = LeakyReLU(0.2)(Dense(hidden, kernel_initializer=init)(in_lat))
    h = LeakyReLU(0.2)(Dense(seq_len * hidden, kernel_initializer=init)(h))
    h = Dropout(0.3)(Reshape((seq_len, hidden))(h))
    base = TimeDistributed(Dense(1, kernel_initializer=init))(h)
    base = Reshape((1, seq_len))(Flatten()(base))

    # control branch (insulin / carb heads)
    c = Concatenate()([Flatten()(in_ins), Flatten()(in_carb)])
    c = LeakyReLU(0.2)(Dense(64, kernel_initializer=init)(c))
    w_raw = Reshape((seq_len, 2))(Dense(seq_len * 2, kernel_initializer=init)(c))
    w_raw = LSTM(16, return_sequences=True)(w_raw)
    w_raw = Dense(2)(w_raw)
    w_ins_raw, w_carb_raw = w_raw[..., 0], w_raw[..., 1]

    w_carb =  (K.softplus(w_carb_raw) + MIN_GAIN_MGDL)   # given: always positive -> raises BG
    # ---- BLANK 3 ----
    w_ins  = -(K.softplus(w_carb_raw) + MIN_GAIN_MGDL)           # hint: negate (softplus(w_ins_raw) + MIN_GAIN_MGDL)

    w_ins  = Reshape((1, seq_len))(w_ins)
    w_carb = Reshape((1, seq_len))(w_carb)
    ins  = Reshape((1, 1))(Flatten()(in_ins))
    carb = Reshape((1, 1))(Flatten()(in_carb))
    control = CONTROL_SCALE * (ins * w_ins + carb * w_carb)

    return Model([in_lat, in_ins, in_carb], base + control, name="decoder_controlled")

decoder = define_decoder(); decoder.summary()

Model: "decoder_controlled"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 in_ins (InputLayer)         [(None, 1, 1)]               0         []                            
                                                                                                  
 in_carb (InputLayer)        [(None, 1, 1)]               0         []                            
                                                                                                  
 flatten_1 (Flatten)         (None, 1)                    0         ['in_ins[0][0]']              
                                                                                                  
 flatten_2 (Flatten)         (None, 1)                    0         ['in_carb[0][0]']             
                                                                                 

Here's the decoder architecture you just built — the two branches and the sign asymmetry that guarantees monotonicity:

## 3 · The loss — reconstruction + KL

The trainer adds MSE reconstruction, the KL term, and a soft counterfactual monotonicity penalty. You write the KL divergence between the approximate posterior and the standard normal prior.

<div style="border-left:4px solid #1D9E75;background:#E1F5EE;padding:14px 18px;border-radius:0 8px 8px 0">
<div style="display:flex;align-items:center;gap:10px;margin-bottom:6px">
<span style="background:#0F6E56;color:#E1F5EE;font-size:11px;font-weight:500;letter-spacing:.04em;padding:3px 9px;border-radius:20px">BLANK 4</span>
<span style="color:#04342C;font-size:15px;font-weight:500">KL divergence</span>
<span style="margin-left:auto;color:#0F6E56;font-size:12px">Notebook A · loss</span>
</div>
<p style="margin:6px 0 0;color:#063d31;font-size:13.5px;line-height:1.6">For a Gaussian posterior vs. a standard-normal prior, the closed-form KL per latent dim is <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">-0.5 * (1 + log_var - mean^2 - exp(log_var))</code>, summed over the latent dimension and averaged over the batch.</p>
<p style="margin:8px 0 0;color:#0F6E56;font-size:13.5px"><b style="font-weight:500">Your task:</b> complete the KL using <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">z_mean</code> and <code style="background:#e8e6dd;padding:1px 5px;border-radius:4px">z_log_var</code>.</p>
</div>

In [8]:
from tensorflow.keras.optimizers import Adam
from twin_workshop import (KL_WEIGHT, MONO_WEIGHT, LR, INSULIN_UP, CARBS_UP, K_INS_100, K_CARB_100)

class VAETrainer(tf.keras.Model):
    def __init__(self, encoder, decoder, **kw):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder
        self.kl_weight, self.mono_weight = KL_WEIGHT, MONO_WEIGHT
        self.insulin_up, self.carbs_up = INSULIN_UP, CARBS_UP
        self.k_ins_100, self.k_carb_100 = K_INS_100, K_CARB_100

    def train_step(self, data):
        bg, ins, carb = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder([bg, ins, carb], training=True)
            y0 = self.decoder([z, ins, carb], training=True)
            recon = K.mean(K.square(bg - y0))

            # ---- BLANK 4 ----
            kl = -0.5 * K.mean(K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1))   # hint: 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)

            y_ins_up  = self.decoder([z, ins * self.insulin_up, carb], training=True)
            y_carb_up = self.decoder([z, ins, carb * self.carbs_up], training=True)
            mu_base, mu_i, mu_c = (K.mean(y0, axis=[1,2]), K.mean(y_ins_up, axis=[1,2]),
                                   K.mean(y_carb_up, axis=[1,2]))
            m_i = (self.insulin_up - 1.0) * self.k_ins_100
            m_c = (self.carbs_up  - 1.0) * self.k_carb_100
            mono = K.mean(K.relu(mu_i - (mu_base - m_i)) + K.relu((mu_base + m_c) - mu_c))
            total = recon + self.kl_weight * kl + self.mono_weight * mono
        grads = tape.gradient(total, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"loss": total, "recon": recon, "kl": kl, "mono": mono}

vae = VAETrainer(encoder, decoder); vae.compile(optimizer=Adam(LR))
print("VAE trainer ready.")

VAE trainer ready.


<!-- A3 — after cell 17 -->
<div style="border:1.5px dashed #0F6E56;border-radius:14px;background:#FDF0EA;padding:18px 22px;margin:16px 0">
<div style="display:flex;align-items:center;gap:9px;margin-bottom:10px">
<span style="width:22px;height:22px;border-radius:50%;background:#0F6E56;color:#E1F5EE;font-size:12px;font-weight:700;display:inline-flex;align-items:center;justify-content:center;flex-shrink:0">?</span>
<span style="color:#0F6E56;font-size:11px;font-weight:600;letter-spacing:.09em;text-transform:uppercase">Look back at what you built</span>
</div>
<p style="margin:0;color:#04342C;font-size:15.5px;font-weight:600;font-style:italic;line-height:1.45">Two safeguards — same job, or different jobs?</p>
<p style="margin:10px 0 0;color:#063d31;font-size:13.5px;line-height:1.65">The decoder's <code style="background:#cfe8de;padding:1px 5px;border-radius:4px">w_ins</code> is architecturally forced negative — that's absolute, no training required. But the loss you just completed <i>also</i> adds a <code style="background:#cfe8de;padding:1px 5px;border-radius:4px">mono</code> penalty comparing outputs under scaled-up insulin/carbs to baseline. If the sign is already guaranteed, what is <code style="background:#cfe8de;padding:1px 5px;border-radius:4px">mono</code> protecting against that the hard constraint doesn't?</p>
<p style="margin:14px 0 0;padding-top:12px;border-top:1px dashed #0F6E56;color:#0F6E56;font-size:13.5px;line-height:1.5"><b>Hint →</b> the sign constraint guarantees direction. Look at what <code style="background:#cfe8de;padding:1px 5px;border-radius:4px">m_i</code> and <code style="background:#cfe8de;padding:1px 5px;border-radius:4px">m_c</code> represent — what property beyond direction are they enforcing?</p>
</div>

## 4 · Confirm against the shipped decoder

Your decoder has the **same architecture** as the population-pretrained one. Load
the provided weights into it and run a forward pass — if shapes line up, you've
built the right model. (You'll personalize it in Notebook B.)

In [9]:
loaded = load_model(PRETRAINED, compile=False)
decoder.set_weights(loaded.get_weights())
z = np.zeros((2, LATENT_DIM), "float32"); s = np.zeros((2, 1, 1), "float32")
print("forward pass output shape:", decoder.predict([z, s, s], verbose=0).shape)
print("Architecture matches the shipped decoder. Continue in Notebook B.")

forward pass output shape: (2, 1, 18)
Architecture matches the shipped decoder. Continue in Notebook B.
